In [ ]:
import h5py
from os.path import join
import os
from scipy.spatial.transform.rotation import Rotation
import numpy as np
from matplotlib import pyplot
from PIL import Image
import cv2
import tqdm
import collections
from importlib import reload

In [ ]:
from IPython.display import display

In [ ]:
import vis
from datasets.preprocessing import imread, extract_image_roi, imencode, imdecode
from utils import iter_batched

In [ ]:
# DATAFILENAME = '/media/mwelter/Stuff/head-tracking-datasets/affectnet_train_decafit.h5'
# OUTFILENAME = '/media/mwelter/Stuff/head-tracking-datasets/affectnet_train.h5'
# IMAGEFOLDER = '/mnt/BigData/head-tracking-datasets/affectnet/train_set/'
# ENABLE_IDENTITIES = False
# SCALE_AND_CROP = False

DATAFILENAME = '/media/mwelter/Stuff/head-tracking-datasets/vggface2decafit.h5'
OUTFILENAME = '/media/mwelter/Stuff/head-tracking-datasets/vggface2-tracker.h5'
IMAGEFOLDER = '/mnt/BigData/head-tracking-datasets/VGG-Face2/train/imgs/'
ENABLE_IDENTITIES = True
SCALE_AND_CROP = True

hdf5_imagebuffer_dtype = h5py.special_dtype(vlen=np.dtype('uint8'))
CHUNKSIZE = 1024

In [ ]:
def grab_identity(dsitem):
    id_, _ = dsitem.decode('ascii').split('/')
    return id_

def generate_sequence_starts(identities):
    last = None
    occured = set()
    
    def checked(id_):
        nonlocal last
        nonlocal occured
        if id_ != last and id_ in occured:
            raise RuntimeError(f"Id {id_} already in encountered in another sequence. Ids must be ordered so that each appears in one contiguous block of samples")
        last = id_
        occured.add(id_)
        return id_

    identities = [ checked(id_) for id_ in identities ]
    _, indices = np.unique(identities, return_index=True)
    reordering = np.argsort(indices)
    # Append the number of items to get the end of the last sequence.
    return np.hstack([indices[reordering], [len(identities)]])

# FIXME: Rather specific to VGGFace2
def create_sequence_start_dataset(sourcefile, destinationfile):
    identities_seq = (grab_identity(item) for item in tqdm.tqdm(sourcefile['fits/filename']))
    sequence_starts = generate_sequence_starts(identities_seq)
    destinationfile.create_dataset('sequence_starts', data = sequence_starts)

In [ ]:
def copy_flame_parameters(sourcefile, destinationfile):
    flamegroup = destinationfile.require_group('flame')
    for key in ['exp', 'light', 'pose', 'shape', 'tex']:
        sourcefile.copy(sourcefile['fits'][key], flamegroup, key)

In [ ]:
def iter_batched(iterable, batchsize):
    it = iter(iterable)
    while True:
        ret = [*zip(range(batchsize),it)]
        ret = [ x for _,x in ret ]
        if not ret:
            break
        yield ret

        
def convert_type_for_storage(name, data):
    if name in ('image','images'):
        v = [ imencode(img) for img in data ]
        return v, hdf5_imagebuffer_dtype, (len(v),)
    else:
        data = np.array(data)
        if data.dtype in (np.uint8, np.int32, np.float16):
            v = data
        elif data.dtype in (np.float32, np.float64):
            v = data.astype(np.float16)
        elif isinstance(data[0], str):
            assert all(len(s)<32 for s in data)
            v = np.array([x.encode('ascii') for x in data], dtype = 'S32')
        else:
            assert False, "Bad data "+str(data)+" of type "+str(data.dtype)
        return v, v.dtype, v.shape

    
def save_batch_to_hdf5(f, batch, start_idx, max_size):
    chunksize = min(max_size, CHUNKSIZE)
    for name,v in batch.items():
        v, hdf5dtype, shape = convert_type_for_storage(name,v)
        if name not in f:
            assert start_idx==0
            ds = f.create_dataset(name, shape=(max_size,)+shape[1:], dtype=hdf5dtype, chunks=(chunksize,)+shape[1:])
        else:
            ds = f[name]
        ds[start_idx:start_idx+shape[0]] = v

In [ ]:
class Converter(object):
    def __init__(self, datafile, imagefolder, range_, cropsize=256, roi_expansionfactor=1.5, monochrome = True):
        self.datafile = datafile
        self.fitgroup = self.datafile['fits']
        self.range_ = range_
        self.imagefolder = imagefolder
        self.cropsize = cropsize
        self.roi_expansionfactor = roi_expansionfactor
        self.monochrome = monochrome

    def convert_points_and_pose(self):
        '''
            Returns position, size, and landmarks relative to the original images pixel coordinates
        '''
        poses, cams, landmarks3d, landmarks2d, crop, head_box, joint_positions = [np.asarray(self.fitgroup[k][self.range_,...]) for k in 'pose cam landmarks3d landmarks2d crop head_box joint_positions'.split()]

        # Landmarks are orthographically projected in the range [-1, 1]
        # crop is x, y, size
        # cam is scale then center ... hm ... a little inconsistent. FIXME!

        crop_center = crop[:,:2]
        crop_size = crop[:,2]

        head_size_meters = 0.1

        # The global coordinate system is different from DECA. Here X points to the viewer
        # and Z goes right. In DECA Z points to the viewer and X goes left. So in the next
        # couple of lines a coordinate transformation is done R' = P R Pt to transform the
        # rotation into the new coordinate system.
        rots = Rotation.from_rotvec(poses[:,:3])  
        P = np.array([
            [ 0, 0, 1  ],
            [ 0, 1, 0  ],
            [ -1, 0, 0  ]
        ], dtype=np.float64)[None,...]
        rots = Rotation.from_matrix(np.matmul(P, np.matmul(rots.as_matrix(), P.transpose(0,2,1))))
        quats = rots.as_quat()

        landmarks3d_vis = landmarks3d[...,3].copy()
        landmarks3d = landmarks3d[...,:3].copy()
        landmarks2d = landmarks2d.copy()
        head_box = np.reshape(head_box,(-1,2,2)).copy()
        head_center = joint_positions[:,1,:2].copy()
        
        eye_corner_indices = [45, 42, 39, 36]
        between_eyes = np.average(landmarks3d[:,eye_corner_indices,:], axis=1)
        landmarks3d[:,:,2] -= between_eyes[:,None,2]

        head_box *= crop_size[:,None,None]*0.5
        head_box += crop_center[:,None]
        head_center *= crop_size[:,None]*0.5
        head_center += crop_center
        
        scale = cams[:,0]*crop_size*head_size_meters*0.5
        #pos = between_eyes[:,:2]*crop_size[:,None]*0.5 + crop_center
        coord = np.concatenate([head_center,scale[:,None]],axis=1)

        landmarks3d *= crop_size[:,None,None]*0.5
        landmarks3d[:,:,:2] += crop_center[:,None,:]

        landmarks2d *= crop_size[:,None,None]*0.5
        landmarks2d += crop_center[:,None,:]

        return {
            'pose' : quats,
            'coord' : coord,
            'pt3d_68' : landmarks3d,
            'head_box' : head_box.reshape(-1,4),
            'pt3d_68_vis' : landmarks3d_vis
        }
    
    @staticmethod
    def _offset_scale_points(pts : np.ndarray, offset, scale):
        if pts.ndim==3:
            offset = offset[:,None]
            scale = scale[:,None]
        pts = pts.copy()
        pts[...,:2] = (pts[...,:2]+offset)*scale[:,None]
        if pts.shape[-1]==3:
            pts[...,2] = pts[...,2]*scale
        return pts
    
    def generate_batch_no_crop(self, pose, coord, pt3d_68, head_box, pt3d_68_vis):
        def load_image(idx):
            return cv2.cvtColor(imread(join(self.imagefolder,self.fitgroup['filename'][idx].decode('ascii'))), cv2.COLOR_BGR2GRAY if self.monochrome else cv2.COLOR_BGR2RGB)
        images = [ load_image(idx) for idx in self.range_ ]
        roi = np.asarray(self.fitgroup['box'][self.range_,...])
        return {
            'image' : images,
            'pose' : pose,
            'coord' : coord,
            'pt3d_68' : pt3d_68,
            'pt3d_68_vis' : pt3d_68_vis,
            'roi' : roi,
            'head_box' : head_box
        }
        
    
    def generate_crops(self, pose, coord, pt3d_68, head_box, pt3d_68_vis):
        crop = []
        offsets = []
        sizes = []
        roi = np.asarray(self.fitgroup['box'][self.range_,...])
        for my_roi, idx in zip(roi, self.range_):
            img = cv2.cvtColor(imread(join(self.imagefolder,self.fitgroup['filename'][idx].decode('ascii'))), cv2.COLOR_BGR2GRAY if self.monochrome else cv2.COLOR_BGR2RGB)
            img, offset = extract_image_roi(img, my_roi, padding_fraction=self.roi_expansionfactor-1., square=True, return_offset=True)
            sizes.append(img.shape[0])
            img = cv2.resize(img, (self.cropsize, self.cropsize), interpolation = cv2.INTER_AREA)
            crop.append(img)
            offsets.append(offset)
        offset = np.stack(offsets)
        sizes = np.stack(sizes)
        coord = self._offset_scale_points(coord, offset, self.cropsize/sizes)
        pt3d_68 = self._offset_scale_points(pt3d_68, offset, self.cropsize/sizes)
        roi = self._offset_scale_points(roi.reshape(-1,2,2), offset, self.cropsize/sizes).reshape(-1,4)
        head_box = self._offset_scale_points(head_box.reshape(-1,2,2), offset, self.cropsize/sizes).reshape(-1,4)
        return {
            'image' : crop,
            'pose' : pose,
            'coord' : coord,
            'pt3d_68' : pt3d_68,
            'pt3d_68_vis' : pt3d_68_vis,
            'roi' : roi,
            'head_box' : head_box
        }

    def get_identities(self):
        for encodedfilename in self.fitgroup['filename'][self.range_]:
            filename = encodedfilename.decode('ascii')
    
    def make_batch(self):
        data = self.convert_points_and_pose()
        if SCALE_AND_CROP:
            data = self.generate_crops(**data)
        else:
            data = self.generate_batch_no_crop(**data)
        # Translate keys, because I'm an idiot
        data = {
            'images' : data['image'],
            'quats' : data['pose'],
            'pt3d_68' : data['pt3d_68'].transpose(0,2,1), # Because for some stupid reason I put the coordinate index before the point index ...
            'rois' : data['roi'],
            'head_rois' : data['head_box'].reshape(-1,4),
            'pt3d_68_vis' : data['pt3d_68_vis'],
            'coords' : data['coord']
        }
        return data
        
    def save_to_hdf5(self, destination, start_idx, max_size):
        batch = self.make_batch()
        save_batch_to_hdf5(destination, batch, start_idx, max_size)


with h5py.File(DATAFILENAME,'r') as datafile:
    cvt = Converter(datafile,IMAGEFOLDER,range(100))
    data = cvt.make_batch()

def display_data(idx):
    #img = imread('/mnt/BigData/head-tracking-datasets/VGG-Face2/train/imgs/'+datafile['fits/filename'][idx].decode('ascii'))
    #img = imdecode(data['image'][idx], color=True)
    img = cv2.cvtColor(data['images'][idx], cv2.COLOR_GRAY2RGB)
    vis.draw_axis(img, Rotation.from_quat(data['quats'][idx]), data['coords'][idx,0], data['coords'][idx,1], data['coords'][idx,2]*0.5)
    vis.draw_points3d(img, data['pt3d_68'][idx], labels=False)
    vis.draw_roi(img, data['rois'][idx], (255,0,255), 1)
    vis.draw_roi(img, data['head_rois'][idx], (255,255,255), 1)
    display(Image.fromarray(img))
    
for idx in range(30):
    display_data(idx)

In [ ]:
with h5py.File(DATAFILENAME, 'r') as datafile, h5py.File(OUTFILENAME, 'w') as outputfile:
    num_samples = datafile['fits/filename'].shape[0]
    if ENABLE_IDENTITIES:
        print ("Generating sequence starts ...")
        create_sequence_start_dataset(datafile, outputfile)
    #print ("Copying flame parameters")
    #copy_flame_parameters(datafile, outputfile)
    print ("Converting pose data, landmarks and cropping images ...")
    for indices in iter_batched(tqdm.trange(num_samples), 128):
        cvt = Converter(datafile,IMAGEFOLDER,indices)
        cvt.save_to_hdf5(outputfile, start_idx=indices[0], max_size=num_samples)